# 00 · Setup de ElectroCasa Lakehouse
Crea catálogos `electrocasa_dev` y `electrocasa_prod`, schemas `bronze`/`silver`/`gold`, el volume `vol_landing` con una subcarpeta por fuente,
las funciones de column mask que usa `silver.empleados`, y demuestra `GRANT` / `REVOKE` con `SHOW GRANTS` antes y después.
Es idempotente: se puede ejecutar varias veces.

In [ ]:
CATALOGOS = ["electrocasa_dev", "electrocasa_prod"]
ESQUEMAS = ["bronze", "silver", "gold"]
FUENTES = ["ventas", "catalogo", "empleados", "resenas", "devoluciones", "sucursales"]  # tracking_envios llega por Lakeflow Connect

# Free Edition sin poder crear los grupos: poner USAR_GRUPO_EN_MASCARA = False y tu correo.
# La máscara compara entonces con current_user() en vez de con el grupo (documentarlo en el README).
USAR_GRUPO_EN_MASCARA = True
CORREO_ALTERNATIVO = ""

G_INGENIERIA = "electrocasa_ingenieria"
G_ANALISTAS = "electrocasa_analistas"
G_AUDITORIA = "electrocasa_auditoria"

def sql(texto):
    print(texto.strip())
    return spark.sql(texto)

def sql_tolerante(texto):
    # Ejecuta y, si falla (grupo inexistente, sin permiso sobre system...), imprime el error completo para el README.
    try:
        return sql(texto)
    except Exception as e:
        print(f"  ERROR -> {type(e).__name__}: {str(e)[:600]}\n")
        return None

## 1. Catálogos, schemas y volume de landing

In [ ]:
for cat in CATALOGOS:
    sql(f"CREATE CATALOG IF NOT EXISTS {cat}")
    for esquema in ESQUEMAS:
        sql(f"CREATE SCHEMA IF NOT EXISTS {cat}.{esquema}")
    sql(f"CREATE VOLUME IF NOT EXISTS {cat}.default.vol_landing")
    for fuente in FUENTES:
        dbutils.fs.mkdirs(f"/Volumes/{cat}/default/vol_landing/{fuente}")
    print(f"OK {cat}: schemas {ESQUEMAS} + volume vol_landing con {len(FUENTES)} subcarpetas\n")

## 2. Column masks para `silver.empleados`
Solo `electrocasa_ingenieria` ve el valor real:
* `mask_dni`: los demás ven `****` + últimos 4 dígitos (`****1234`).
* `mask_salario`: los demás ven `NULL`.

El pipeline las referencia con `MASK` en el schema de la tabla, por eso deben existir antes de la primera ejecución.

In [ ]:
if USAR_GRUPO_EN_MASCARA:
    ES_INGENIERIA = f"is_account_group_member('{G_INGENIERIA}')"
else:
    assert CORREO_ALTERNATIVO, "Define CORREO_ALTERNATIVO cuando no uses el grupo"
    ES_INGENIERIA = f"current_user() = '{CORREO_ALTERNATIVO}'"

for cat in CATALOGOS:
    sql(f"""
        CREATE OR REPLACE FUNCTION {cat}.silver.mask_dni(dni STRING)
        RETURNS STRING
        RETURN CASE WHEN dni IS NULL THEN NULL
                    WHEN {ES_INGENIERIA} THEN dni
                    ELSE '****' || right(dni, 4) END
    """)
    sql(f"""
        CREATE OR REPLACE FUNCTION {cat}.silver.mask_salario(salario DECIMAL(12,2))
        RETURNS DECIMAL(12,2)
        RETURN CASE WHEN {ES_INGENIERIA} THEN salario ELSE NULL END
    """)

## 3. Permisos por grupo
Los grupos se crean antes desde **Settings > Identity and access > Groups** (no con API/CLI/SDK: quedarían como grupos locales y el `GRANT` falla con `PRINCIPAL_DOES_NOT_EXIST`). Agrégate a `electrocasa_ingenieria`.

| Grupo | Acceso |
|---|---|
| `electrocasa_ingenieria` | Todo el catálogo (lectura/escritura/creación) y el volume de landing |
| `electrocasa_analistas` | Solo `gold` (lectura) |
| `electrocasa_auditoria` | Solo lectura sobre `gold` y acceso a la evidencia de auditoría y lineage (`system.access`) |

In [ ]:
for cat in CATALOGOS:
    sql(f"GRANT USE CATALOG, USE SCHEMA, SELECT, MODIFY, CREATE TABLE, CREATE VOLUME, CREATE FUNCTION, CREATE MATERIALIZED VIEW ON CATALOG {cat} TO `{G_INGENIERIA}`")
    sql(f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME {cat}.default.vol_landing TO `{G_INGENIERIA}`")

    sql(f"GRANT USE CATALOG ON CATALOG {cat} TO `{G_ANALISTAS}`")
    sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {cat}.gold TO `{G_ANALISTAS}`")

    sql(f"GRANT USE CATALOG ON CATALOG {cat} TO `{G_AUDITORIA}`")
    sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {cat}.gold TO `{G_AUDITORIA}`")

## 3b. Evidencia de auditoría (`system.access`)
Auditoría necesita `USE CATALOG` sobre `system`, `USE SCHEMA` sobre `system.access` y `SELECT` sobre `audit`, `table_lineage` y `column_lineage`.
En Free Edition **no se pueden otorgar** (ni el administrador tiene `MANAGE` sobre `system`): el error queda impreso para pegarlo en el README y la evidencia
pasa a la pestaña **Lineage** de Catalog Explorer.

In [ ]:
sql_tolerante(f"GRANT USE CATALOG ON CATALOG system TO `{G_AUDITORIA}`")
sql_tolerante(f"GRANT USE SCHEMA ON SCHEMA system.access TO `{G_AUDITORIA}`")
for tabla in ("audit", "table_lineage", "column_lineage"):
    sql_tolerante(f"GRANT SELECT ON TABLE system.access.{tabla} TO `{G_AUDITORIA}`")

## 4. Demostración de `REVOKE` (SHOW GRANTS antes y después)

In [ ]:
CAT_DEMO = "electrocasa_dev"

# Concesión temporal: se le da a analistas acceso a silver para poder revocarlo
sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {CAT_DEMO}.silver TO `{G_ANALISTAS}`")

print("\n=== ANTES del REVOKE: SHOW GRANTS en silver ===")
display(sql(f"SHOW GRANTS ON SCHEMA {CAT_DEMO}.silver"))

In [ ]:
sql(f"REVOKE USE SCHEMA, SELECT ON SCHEMA {CAT_DEMO}.silver FROM `{G_ANALISTAS}`")

print("\n=== DESPUÉS del REVOKE: SHOW GRANTS en silver (analistas ya no aparece) ===")
display(sql(f"SHOW GRANTS ON SCHEMA {CAT_DEMO}.silver"))

In [ ]:
for grupo in (G_INGENIERIA, G_ANALISTAS, G_AUDITORIA):
    print(f"\n=== Privilegios efectivos de {grupo} sobre {CAT_DEMO} ===")
    display(sql(f"SHOW GRANTS `{grupo}` ON CATALOG {CAT_DEMO}"))

## 5. Siguientes pasos
1. Subir los archivos de cada fuente a `/Volumes/<catalogo>/default/vol_landing/<fuente>/`.
2. Crear la Connection de Unity Catalog hacia Azure SQL (`conn_azure_sql_electrocasa`) desde *Catalog → Connections*.
3. `databricks bundle deploy -t dev` y `databricks bundle run electrocasa_job -t dev`.